classification binaire de base
    Créer un dataset (toutes les images livrable 1)
    Load un modèle préentrainé


Récupérer les index des photos du dataset


Effectuer le denoizing sur le dataset prédit



In [ ]:
import tensorflow as tf
import os
import glob
import matplotlib.pyplot as plt
import math


folders = [
    "/tf/projet/Dataset/Sketch",
    "/tf/projet/Dataset/Photo",
    "/tf/projet/Dataset/Painting",
    "/tf/projet/Dataset/Schematics",
    "/tf/projet/Dataset/Text",
]

# Lister toutes les images
image_paths = []
for folder in folders:
    image_paths.extend(glob.glob(os.path.join(folder, "*.[jp][pn]g")))

def load_and_preprocess(path):
    image = tf.io.read_file(path)
    image = tf.image.decode_image(image, channels=3)
    image.set_shape([180, 180, 3])  # Explicitly set the shape
    image = tf.image.resize(image, [180, 180])  # À adapter à ton modèle
    image = tf.cast(image, tf.float32)
    return image, path

dataset = tf.data.Dataset.from_tensor_slices(image_paths)
dataset = dataset.map(load_and_preprocess, num_parallel_calls=tf.data.AUTOTUNE)
dataset = dataset.shuffle(buffer_size=math.ceil(len(image_paths)/4))


# show first elements of dataset
for image, path in dataset.take(1):
    image = image / 255.0
    print(path.numpy())
    print(image.numpy().shape)
    plt.imshow(image.numpy())
    plt.title(path.numpy())
    plt.show()


In [ ]:
# Chargement du modèle de classification
model_classif = tf.keras.models.load_model("/tf/projet/Leyanda_Project/models/saved/Leyanda_CNN_bin_Photo_transfer_e10_b64_class_weight.keras")

BATCH_SIZE = 64

# On batch le dataset
batched_dataset = dataset.batch(BATCH_SIZE)

photo_images_paths = []

for batch_images, batch_paths in batched_dataset:
    predictions = model_classif.predict(batch_images)  # shape (batch_size, 1)
    for i, prob in enumerate(predictions):
        if prob[0] > 0.5:
            photo_images_paths.append(batch_paths[i].numpy())

In [ ]:
# show les  10 dernières images de photo_images_paths
# print(photo_images_paths)
# for path in photo_images_paths:
#     image = tf.io.read_file(path)
#     image = tf.image.decode_image(image, channels=3)
#     image = tf.cast(image, tf.float32)
#     image = image / 255.0
#     plt.imshow(image.numpy())
#     plt.title(path)
#     plt.show()

In [ ]:
# Chargement du modèle de débruitage (autoencodeur)
denoiser = tf.keras.models.load_model("/tf/projet/Leyanda_Project/models/saved/denoiser.keras")

BATCH_SIZE = 32

# Dataset d'images "photo"
photo_dataset = tf.data.Dataset.from_tensor_slices(photo_images_paths)
photo_dataset = photo_dataset.map(load_and_preprocess, num_parallel_calls=tf.data.AUTOTUNE)
photo_dataset = photo_dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

# Chargement du modèle
denoiser = tf.keras.models.load_model("/tf/projet/Leyanda_Project/models/saved/denoiser.keras")

# Dossier de sortie
output_dir = "/tf/projet/Dataset_Filtered_Denoised"
if not os.path.exists(output_dir):
    os.makedirs(output_dir)
else:
    for f in os.listdir(output_dir):
        os.remove(os.path.join(output_dir, f))

# Appliquer le débruitage par batch
for batch_images, batch_paths in photo_dataset:
    denoised_batch = denoiser.predict(batch_images, verbose=0)
    
    for img, path in zip(denoised_batch, batch_paths.numpy()):
        img = tf.clip_by_value(img, 0.0, 255.0)  # Ensure pixel values are in the range [0, 255]
        img = tf.cast(img, dtype=tf.uint8)  # Convert to uint8
        encoded_img = tf.image.encode_jpeg(img)  # Encode as JPEG
        
        filename = os.path.basename(path.decode('utf-8'))
        out_path = os.path.join(output_dir, filename)
        tf.io.write_file(out_path, encoded_img)

In [ ]:
import tensorflow as tf
import os

# Chemin de l'image à tester
test_image_path = "/tf/projet/Dataset_Filtered_Denoised"
first_image = sorted(os.listdir(test_image_path))[0]
image_path = os.path.join(test_image_path, first_image)

# Chargement de l'image
image = tf.io.read_file(image_path)
image = tf.image.decode_jpeg(image, channels=3)
image = tf.image.resize(image, [180, 180])  # Taille à adapter selon ton modèle
image = tf.keras.applications.inception_v3.preprocess_input(image)  # Exemple avec Inception
image = tf.expand_dims(image, axis=0)  # Batch dimension

# Chargement du modèle de captioning
caption_model = tf.keras.models.load_model("/tf/projet/Leyanda_Project/models/saved/Leyanda_captioning_model_20250423_192048.keras")

# Génération de la légende
# Assuming the second input is a sequence of tokens (e.g., a start token for caption generation)
# Replace `start_token` with the actual start token or input expected by your model
start_token = tf.constant([[0]])  # Example: a batch of start tokens with value 0
predicted_caption = caption_model.predict([image, start_token])
# post-traitement ici dépend de ton modèle (ex: décodage des tokens)
# par exemple, si tu utilises un tokenizer :
# caption = tokenizer.decode(predicted_caption)

print("Légende prédite (brute) :", predicted_caption)